## 10-714: 作业 0

本次作业的目的是让你快速了解在学习本课程之前应该熟悉的一些概念和思想。作业要求你构建一个基本的 softmax 回归算法，以及一个简单的两层神经网络。你需要同时用原生 Python（使用 numpy 库）和（针对 softmax 回归）原生 C/C++ 来实现。本次作业还会带你了解如何将作业提交到我们的自动评分系统。在此过程中，我们会给出一些关于如何实现不同函数的指导，但整体细节由你决定。不过需要说明的是，在 Python 版本中，你应该大量使用 numpy 中的线性代数调用：如果尝试使用显式循环，通常会导致代码比预期的慢得多。

**我们知道这份作业中有很多文字说明，尤其是在开头部分，而实际的编码部分相对较少。但即便如此，请仔细阅读这份说明的全部内容。这样做将帮助你了解我们设计作业的流程和理念，对你完成后续作业会有很大帮助。**

10-714 课程的所有作业代码开发都可以在 Google Colab 环境中完成。不过，除了在 Colab 笔记本中直接使用代码块外，你需要开发的大部分代码都会写入 `.py` 文件中，这些文件会（自动）下载到你的 Google Drive，而你主要使用笔记本运行 shell 脚本来测试代码并提交给自动评分系统（也可以在开发过程中测试代码片段，但这并非必须）。这种使用 Colab 笔记本的方式与通常不太一样（通常人们更多将其当作交互式编码环境，直接在笔记本中编写代码单元）。但我们这样做的理由其实很简单：Colab 不仅是一个好用的云端笔记本环境，还提供了非常方便的途径来访问“标准”的云端 GPU 系统，你可以快速启动这些系统，从而在开发后期（尤其是基于 CUDA 的代码）时无需物理 GPU，也无需自己配置 CUDA 库。话虽如此，**你完全可以在任何自己偏好的环境中进行开发和提交**，我们只是无法保证对 Colab 之外的环境提供支持。

开始之前，请**复制此笔记本**：通过“文件”菜单选择“在云端硬盘中保存副本”，然后运行下面的代码块。这会将你的 Google Drive 文件夹挂载到 Colab 笔记本环境中，创建一个 `/10714/hw0` 目录，并将 HW0 公开仓库克隆到该目录中。

In [ ]:
# Code to set up the assignment
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/
!mkdir -p 10714
%cd /content/drive/MyDrive/10714
!git clone https://github.com/dlsyscourse/hw0.git
%cd /content/drive/MyDrive/10714/hw0

This next cell will then install the libraries required.

In [2]:
!pip3 install --upgrade --no-deps git+https://github.com/dlsyscourse/mugrade.git
!pip3 install pybind11
!pip3 install numdifftools

  Cloning https://github.com/dlsyscourse/mugrade.git to /tmp/pip-req-build-rejj8c2x
  Running command git clone --filter=blob:none --quiet https://github.com/dlsyscourse/mugrade.git /tmp/pip-req-build-rejj8c2x
  Resolved https://github.com/dlsyscourse/mugrade.git to commit 717e300a5c2ddc0c729746946f8dc9f0d1c0ecea
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 问题 1：一个基础的 `add` 函数，以及测试/自动评分基础

为了说明这些作业的工作流程和自动评分系统，我们将使用一个实现 `add` 函数的简单示例。请注意，上面运行的命令会在你的 `10714/hw0` 目录中创建以下结构：

    data/
        train-images-idx3-ubyte.gz
        train-labels-idx1-ubyte.gz
        t10k-images-idx3-ubyte.gz
        t10k-labels-idx1-ubyte.gz
    src/
        simple_ml.py
        simple_ml_ext.cpp
    tests/
        test_simple_ml.py
    Makefile

`data/` 目录包含本作业所需的数据（MNIST 数据集的一个副本）；`src/` 目录包含你将编写实现的源文件；`tests/` 目录包含用于（本地）评估你的解决方案的测试，同时也可将其提交进行自动评分。而 `Makefile` 文件是一个 Makefile，它会编译代码（与作业中 C++ 部分相关）。

第一个作业问题要求你实现 `simple_ml.add()` 函数（这个简单的函数在别处不会用到，它只是一个示例，旨在让你熟悉作业的结构）。查看 `src/simple_ml.py` 文件，你会找到如下用于 `add()` 函数的函数存根。

```python
def add(x, y):
    """ 一个简单的 'add' 函数，你应该实现它，以熟悉自动评分器和提交流程。
    该问题的答案在作业 notebook 中。

    Args:
        x (Python 数字或 numpy array)
        y (Python 数字或 numpy array)

    Return:
        x + y 的和
    """
    ### YOUR CODE HERE
    pass
    ### END YOUR CODE
```
每个文件中的文档字符串定义了你的函数应该产生的预期输入/输出映射（你需要养成仔细阅读的习惯，因为我们通常发现，提交中错误的头号来源就是没有阅读规范）。希望你很清楚如何实现这个函数。你只需将 `pass` 语句替换为正确的代码，即下面的代码：

```python
def add(x, y):
    """ 一个简单的 'add' 函数，你应该实现它，以熟悉自动评分器和提交流程。
    该问题的答案在作业 notebook 中。

    Args:
        x (Python 数字或 numpy array)
        y (Python 数字或 numpy array)

    Return:
        x + y 的和
    """
    ### YOUR CODE HERE
    return x + y
    ### END YOUR CODE
```
现在请在你的 `src/simple_ml.py` 文件中进行此操作。

### 运行本地测试

现在，你需要测试一下你的代码是否运行正确，如果正确，就将其提交到自动评分系统。在本课程中，我们将使用标准的代码单元测试工具，即 `pytest` 系统。一旦你在 `src/simple_ml.py` 文件中编写了正确的代码，请运行以下命令。

In [4]:
!python3 -m pytest -k "add"

============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0
rootdir: /home/x1879/dlsys/hw0
collected 6 items / 5 deselected / 1 selected                                  

tests/test_simple_ml.py .                                                [100%]

======================= 1 passed, 5 deselected in 0.33s ========================


如果一切正确，您将看到一个测试成功通过。要了解此测试的工作原理，请查看 `tests/test_simple_ml.py` 文件，特别是 `test_add()` 函数：

```python
def test_add():
    assert add(5,6) == 11
    assert add(3.2,1.0) == 4.2
    assert type(add(4., 4)) == float
    np.testing.assert_allclose(add(np.array([1,2]), np.array([3,4])),
                               np.array([4,6]))
```

此代码对您实现的函数运行一套单元测试。如果函数实现正确，那么上述所有断言_应该_全部通过（即代码将无错误执行）。反之，若您的实现有误（例如，将前面的 `x + y` 改成了 `x - y`），则这些断言将失败，而 `pytest` 会指出相应的测试失败。

In [24]:
# in this example cell, we replaced "x + y" with "x - y" in simple_ml.add()
!python3 -m pytest -k "add"

============================= test session starts ==============================
platform darwin -- Python 3.7.3, pytest-4.3.1, py-1.8.0, pluggy-0.9.0
rootdir: /Users/zkolter/Dropbox/class/10-714/homework/hw0, inifile:
plugins: remotedata-0.3.1, openfiles-0.3.2, doctestplus-0.3.0, arraydiff-0.3
collected 6 items / 5 deselected / 1 selected                                  

tests/test_simple_ml.py F                                                [100%]

=================================== FAILURES ===================================
___________________________________ test_add ___________________________________

    def test_add():
>       assert add(5,6) == 11
E       assert -1 == 11
E        +  where -1 = add(5, 6)

tests/test_simple_ml.py:16: AssertionError
==================== 1 failed, 5 deselected in 0.35 seconds ====================


如您所见，您会收到一条错误消息，指出断言失败的那一行代码，您可以根据这条信息回过头来调试您的实现。**您应该逐渐习惯于阅读并跟踪测试文件，以此更好地理解您的实现应当如何工作。**

学会正确开发和使用单元测试对现代软件开发至关重要，本课程的一个次要成果，或许就是希望您能熟悉单元测试在软件开发中的典型用法。当然，这不完全准确，因为在这里，您并不一定需要自己编写测试才能通过题目，但您_应该_熟悉如何阅读我们提供的测试文件，以此理解您的函数应该如何表现。然而，我们_绝对_也鼓励您为自己的实现编写额外的测试，特别是当您发现代码在本地测试中通过了，但提交时似乎仍然失败的情况。

最后一点小提示。如果您习惯通过 print 语句来调试代码，请注意 **pytest 默认会捕获所有输出**。您可以通过向 pytest 传入 `-s` 标志来禁用该行为，从而在任何情况下都让测试显示所有输出。

### 提交给自动评分器

现在你已经通过了单元测试，是时候将你的解决方案提交给自动评分了。在本课程中，我们使用由课程讲师编写的自定义应用程序进行自动评分。要开始自动评分流程，请前往 https://mugrade.org/courses/1?enroll=664EqzbhsV5p99rbBYf0，并通过 Google 登录**使用你的 Andrew 邮箱**进行登录。

创建账户后，点击“Deep Learning Systems”进入课程。在页面右上方附近，点击**显示 API 密钥**并复制关联的密钥。此密钥与你在本课程中的_提交_相关联，任何持有该密钥的人都可以代你提交作业；因此，你不得与任何人分享此密钥，仅将其用于提交你自己的代码。获取密钥后，运行以下命令。

In [5]:
!python3 -m mugrade submit YOUR_GRADER_KEY_HERE hw0 -k "add"

submit
============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0
rootdir: /home/x1879/dlsys/hw0
collected 6 items / 5 deselected / 1 selected                                  

tests/test_simple_ml.py F

=================================== FAILURES ===================================
__________________________________ submit_add __________________________________

pyfuncitem = <Function submit_add>

    @pytest.hookimpl(hookwrapper=True)
    def pytest_pyfunc_call(pyfuncitem):
        ## prior to test, initialize submission
        global _values, _submission_id, _errors
        _values = []
        _errors = 0
        func_name = pyfuncitem.name[7:]
        if os.environ["MUGRADE_OP"] == "submit":
>           _submission_id = start_submission(func_name)
                             ^^^^^^^^^^^^^^^^^^^^^^^^^^^

../../miniconda3/envs/dlsys/lib/python3.13/site-packages/mugrade/mugrade.py:106: 
_ _ _ _ _ _

运行此命令会将你的 `add` 函数提交到 mugrade 自动评分系统。想了解其内部工作原理，可以再次查看 `tests/test_simply_ml.py` 文件，但这次请关注 `test_add` 函数正下方的 `submit_add()` 函数。

```python
def submit_add():
    mugrade.submit(add(1,2))
    mugrade.submit(add(4.5, 3.2))
    mugrade.submit(type(add(3,2)))
    mugrade.submit(add(np.array([1.,2.]), np.array[5,6]))
```

这段代码看起来有点像上面的单元测试，不过这里用 `mugrade.submit()` 调用替代了断言。这些调用会在不同输入上执行 `add` 函数，然后将结果发送到 mugrade 服务器。服务器会将你的函数输出与正确输出（仅存储在服务器上，本地无法获取，因此你无法预先知晓正确答案）进行比对，并相应地更新你该作业的成绩。登录 mugrade 系统后，你可以进入 "Homework 0" 作业查看更新后的分数（如果已在页面中，请按需刷新）。

**重要提示：** 熟悉自动评分系统的同学可能会注意到，mugrade 的运作方式与大多数系统略有不同。在传统自动评分系统中，你的代码需要通过本地测试（幸运的话，有些课程甚至不提供这些测试），然后将代码打包提交至评分系统，后者会在服务器上解包并运行你的代码，针对（你不可见的）测试用例进行验证。Mugrade 则不同：`submit_add` 函数在**你的系统**上运行（例如你正在使用的 Colab 环境），仅将调用**结果**发送至服务器。

这种设计基于一个微妙但关键的考量。本课程要求你开发一个相当复杂的系统——需要在 GPU 等专用硬件上运行代码，可能还需要通过训练真实神经网络架构的长时间测试。无法调试评分测试用例中的代码执行过程，在实际操作中会构成重大挑战，更不用说服务器容量限制以及通常在截止时间前出现的系统卡顿问题。将计算放在本地意味着你能更深入地了解代码在自动评分器测试用例中的实际运行状况，这对调试极为有利。在类似课程的论坛上，**最常见的**帖子莫过于"我的代码通过了所有本地测试，却在自动评分器上失败了"；虽然 mugrade 仍可能出现这种情况，但你至少可以逐步跟踪代码执行过程，查看**具体在哪里**和**如何**失败的。由于服务器只需简单比对输入结果与正确输出，其架构非常轻量，你可以立即获得自动评分器的反馈——即使在提交截止前的最后一刻也是如此。

这种评分系统的**缺点**在于可能存在作弊风险。因为你完全掌控本地执行自动评分测试用例的过程，理论上你完全可以推测出正确答案，直接返回结果而不实际实现所需代码。例如在本作业后续部分，除了初始的 Python 实现外，你还需编写一个 C++ 版本的 softmax 回归。完全可能通过修改自动评分器，让它使用你的 Python 实现而非 C++ 版本来通过测试。话虽如此，**请切勿以任何方式尝试规避自动评分系统**。我们设计该系统的初衷，是真正帮助你更便捷地调试和开发代码，使体验更接近"真实"开发流程，而非纠结于"搞清自动评分器为何无法编译你特定版本的 CUDA 代码"。为避免此类问题，除提交自动评分器测试外，完成提交时你还需要将 `/src` 目录的 `.tar.gz` 压缩包上传至 mugrade 系统。我们相信这不会成为问题，但若**有任何**疑虑，助教始终可以验证你的代码确实产生了相应结果（他们也会使用 MOSS 等标准剽窃检测系统进行审查）。

## 问题2：加载MNIST数据

现在你对自动评分系统已有所了解，可以尝试在 `src/simple_ml.py` 文件中实现下一个函数：`parse_mnist_data()`。以下是来自该文件的函数声明（通常我们不会再次完整走一遍这个过程，但这里会再演示一次）。

```python
def parse_mnist(image_filename, label_filename):
    """ Read an images and labels file in MNIST format.  See this page:
    http://yann.lecun.com/exdb/mnist/ for a description of the file format.

    Args:
        image_filename (str): name of gzipped images file in MNIST format
        label_filename (str): name of gzipped labels file in MNIST format

    Returns:
        Tuple (X,y):
            X (numpy.ndarray[np.float32]): 2D numpy array containing the loaded 
                data.  The dimensionality of the data should be 
                (num_examples x input_dim) where 'input_dim' is the full 
                dimension of the data, e.g., since MNIST images are 28x28, it 
                will be 784.  Values should be of type np.float32, and the data 
                should be normalized to have a minimum value of 0.0 and a 
                maximum value of 1.0 (i.e., scale original values of 0 to 0.0 
                and 255 to 1.0).

            y (numpy.ndarray[dtype=np.uint8]): 1D numpy array containing the
                labels of the examples.  Values should be of type np.uint8 and
                for MNIST will contain the values 0-9.
    """
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE
```

希望你已熟悉这个文档字符串的格式，并对如何实现该函数有了思路。首先，请访问 http://yann.lecun.com/exdb/mnist/ 或这个备用[链接](https://web.archive.org/web/20220509025752/http://yann.lecun.com/exdb/mnist/)（页面底部），了解 MNIST 数据的二进制格式。然后编写一个加载器，读取这类文件，并按照文档字符串中的规范返回 NumPy 数组（如果在实现过程中遇到任何问题，请仔细阅读文档字符串）。我们建议使用 Python 的 `struct` 模块（以及 `gzip` 模块，当然还有 `numpy` 本身）来实现该函数。

当你实现了这个函数后，运行本地单元测试。

In [9]:
!python3 -m pytest -k "parse_mnist"

============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0
rootdir: /home/x1879/dlsys/hw0
collected 6 items / 5 deselected / 1 selected                                  

tests/test_simple_ml.py .                                                [100%]

======================= 1 passed, 5 deselected in 0.68s ========================


And then submit your code to mugrade.

In [ ]:
!python3 -m mugrade submit YOUR_GRADER_KEY_HERE hw0 -k "parse_mnist"

## 问题3：Softmax损失

请实现`src/simple_ml.py`文件中`softmax_loss()`函数所定义的softmax（又称交叉熵）损失。回顾（希望这是复习内容，不过我们也会在9月1日的课上讲解）对于多类输出，当真实类别$y \in \{1,\ldots,k\}$时，softmax损失以logits向量$z \in \mathbb{R}^k$和真实类别$y \in \{1,\ldots,k\}$作为输入，返回如下定义的损失：
\begin{equation}
\ell_{\mathrm{softmax}}(z, y) = \log\sum_{i=1}^k \exp z_i - z_y.
\end{equation}

注意，如其文档字符串所述，`softmax_loss()`函数接受一个形状为_二维数组_的logits（即一批不同样本的$k$维logits），以及对应的一维真实标签数组，输出整批数据的_平均_softmax损失。请注意，要正确实现，你_不应_使用任何循环，而应完全使用numpy的向量化操作进行计算（为明确期望，例如我们可以指出，我们的参考实现仅由一行代码组成）。

注意，对于softmax损失的“真实”实现，你可能希望对logits进行缩放以防止数值溢出，但这里我们不讨论这一点（即使你不考虑这一点，本作业的其余部分也能正常运行）。下述代码将对测试用例进行测试。

In [1]:
!python3 -m pytest -k "softmax_loss"

============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0
rootdir: /home/x1879/dlsys/hw0
collected 6 items / 5 deselected / 1 selected                                  

tests/test_simple_ml.py .                                                [100%]

======================= 1 passed, 5 deselected in 0.95s ========================


Then run the submission.

In [ ]:
!python3 -m mugrade submit YOUR_GRADER_KEY_HERE hw0 -k "softmax_loss"

## 问题4：Softmax回归的随机梯度下降

在本问题中，你将实现（线性）softmax回归的随机梯度下降（SGD）。换句话说，正如9月1日课堂上讨论的，我们考虑一个假设函数，该函数通过以下映射将$n$维输入转换为$k$维logits：
\begin{equation}
h(x) = \Theta^T x
\end{equation}
其中$x \in \mathbb{R}^n$为输入，$\Theta \in \mathbb{R}^{n \times k}$为模型参数。给定数据集$\{(x^{(i)} \in \mathbb{R}^n, y^{(i)} \in \{1,\ldots,k\})\}$，$i=1,\ldots,m$，softmax回归对应的优化问题为：
\begin{equation}
\operatorname*{minimize}_{\Theta} \; \frac{1}{m} \sum_{i=1}^m \ell_{\mathrm{softmax}}(\Theta^T x^{(i)}, y^{(i)}).
\end{equation}

回顾课堂上讲过的内容，线性softmax目标函数的梯度为：
\begin{equation}
\nabla_\Theta \ell_{\mathrm{softmax}}(\Theta^T x, y) = x (z - e_y)^T
\end{equation}

\begin{equation}
z = \frac{\exp(\Theta^T x)}{1^T \exp(\Theta^T x)} \equiv \operatorname{normalize}(\exp(\Theta^T x))
\end{equation}
（即$z$为归一化后的softmax概率），其中$e_y$表示第$y$个单位基向量，即一个全零向量，仅在第$y$个位置为1。

我们也可以使用课堂上学过的更紧凑的表示法。具体来说，令$X \in \mathbb{R}^{m \times n}$表示某个包含$m$个输入的设计矩阵（可以是整个数据集或一个小批量），$y \in \{1,\ldots,k\}^m$为对应的标签向量，并对$\ell_{\mathrm{softmax}}$进行重载以表示平均softmax损失，则有：
\begin{equation}
\nabla_\Theta \ell_{\mathrm{softmax}}(X \Theta, y) = \frac{1}{m} X^T (Z - I_y)
\end{equation}
其中
\begin{equation}
Z = \operatorname{normalize}(\exp(X \Theta)) \quad \text{（按行进行归一化）}
\end{equation}
表示logits矩阵，$I_y \in \mathbb{R}^{m \times k}$为标签$y$对应的独热基向量的拼接。

利用这些梯度，实现`softmax_regression_epoch()`函数，该函数运行一轮SGD（遍历数据集一次），使用指定的学习率/步长`lr`和小批量大小`batch`。如文档字符串所述，你的函数应原地（in-place）修改`Theta`数组。实现后，运行测试。

In [5]:
!python3 -m pytest -k "softmax_regression_epoch and not cpp"

============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0
rootdir: /home/x1879/dlsys/hw0
collected 6 items / 5 deselected / 1 selected                                  

tests/test_simple_ml.py .                                                [100%]

======================= 1 passed, 5 deselected in 0.70s ========================


And then run the submission.

In [ ]:
!python3 -m mugrade submit YOUR_GRADER_KEY_HERE hw0 -k "softmax_regression_epoch and not cpp"

### 使用 softmax 回归训练 MNIST

虽然这不是测试的一部分，但现在你已经编写了这段代码，也可以尝试使用 SGD 训练完整的 MNIST 线性分类器。为此，你可以使用 `src/simple_ml.py` 文件中的 `train_softmax()` 函数（我们已经为你编写好了这个函数，所以你不需要自己写，不过你可以看看它是如何工作的）。

你可以通过以下代码了解其工作方式。供参考，如下所示，我们的实现在 Colab 上运行大约需要 3 秒，并实现了 7.97% 的错误率。

In [6]:
import sys
sys.path.append("src/")
from simple_ml import train_softmax, parse_mnist

X_tr, y_tr = parse_mnist("data/train-images-idx3-ubyte.gz", 
                         "data/train-labels-idx1-ubyte.gz")
X_te, y_te = parse_mnist("data/t10k-images-idx3-ubyte.gz",
                         "data/t10k-labels-idx1-ubyte.gz")

train_softmax(X_tr, y_tr, X_te, y_te, epochs=10, lr=0.2, batch=100)

| Epoch | Train Loss | Train Err | Test Loss | Test Err |
|     0 |    0.35134 |   0.10182 |   0.33588 |  0.09400 |
|     1 |    0.32142 |   0.09268 |   0.31086 |  0.08730 |
|     2 |    0.30802 |   0.08795 |   0.30097 |  0.08550 |
|     3 |    0.29987 |   0.08532 |   0.29558 |  0.08370 |
|     4 |    0.29415 |   0.08323 |   0.29215 |  0.08230 |
|     5 |    0.28981 |   0.08182 |   0.28973 |  0.08090 |
|     6 |    0.28633 |   0.08085 |   0.28793 |  0.08080 |
|     7 |    0.28345 |   0.07997 |   0.28651 |  0.08040 |
|     8 |    0.28100 |   0.07923 |   0.28537 |  0.08010 |
|     9 |    0.27887 |   0.07847 |   0.28442 |  0.07970 |


## 问题5：针对两层神经网络的SGD

现在，您已经为线性分类器编写了SGD，接下来考虑一个简单的两层神经网络的情况。具体来说，对于输入 $x \in \mathbb{R}^n$，我们将考虑一个（不带偏置项的）两层神经网络，其形式为
\begin{equation}
z = W_2^T \mathrm{ReLU}(W_1^T x)
\end{equation}
其中 $W_1 \in \mathbb{R}^{n \times d}$ 和 $W_2 \in \mathbb{R}^{d \times k}$ 表示网络的权重（该网络具有 $d$ 维的隐藏单元），$z \in \mathbb{R}^k$ 表示网络输出的logits。我们仍然使用softmax/交叉熵损失，这意味着我们要求解优化问题
\begin{equation}
\operatorname*{minimize}_{W_1, W_2} \;\; \frac{1}{m} \sum_{i=1}^m \ell_{\mathrm{softmax}}(W_2^T \mathrm{ReLU}(W_1^T x^{(i)}), y^{(i)}).
\end{equation}
或者，采用矩阵 $X \in \mathbb{R}^{m \times n}$ 的批处理形式重载记号，该问题也可写为
\begin{equation}
\operatorname*{minimize}_{W_1, W_2} \;\; \ell_{\mathrm{softmax}}(\mathrm{ReLU}(X W_1) W_2, y).
\end{equation}

利用链式法则，我们可以推导出该网络的反向传播更新（我们将在9月8日的课堂上简要介绍这些内容，但为了便于实现，这里先给出最终形式）。具体地，令
\begin{equation}
\begin{split}
Z_1 \in \mathbb{R}^{m \times d} & = \mathrm{ReLU}(X W_1) \\
G_2 \in \mathbb{R}^{m \times k} & = \operatorname{normalize}(\exp(Z_1 W_2)) - I_y \\
G_1 \in \mathbb{R}^{m \times d} & = \mathrm{1}\{Z_1 > 0\} \circ (G_2 W_2^T)
\end{split}
\end{equation}
其中 $\mathrm{1}\{Z_1 > 0\}$ 是一个二元矩阵，其元素为零或一，取决于 $Z_1$ 中每个对应项是否严格为正；$\circ$ 表示逐元素乘法。那么，目标函数的梯度由下式给出
\begin{equation}
\begin{split}
\nabla_{W_1} \ell_{\mathrm{softmax}}(\mathrm{ReLU}(X W_1) W_2, y) & = \frac{1}{m} X^T G_1  \\
\nabla_{W_2} \ell_{\mathrm{softmax}}(\mathrm{ReLU}(X W_1) W_2, y) & = \frac{1}{m} Z_1^T G_2.  \\
\end{split}
\end{equation}

**注意：** 如果您（在9月8日讲座之前）觉得这些精确公式的细节有些晦涩，不必过于担心。这_只是_标准的两层ReLU网络的反向传播公式：$Z_1$ 项计算“前向”过程，而 $G_2$ 和 $G_1$ 项表示反向传播。但更新公式的具体形式可能会因您所用的神经网络记号、损失的具体构造方式，以及之前是否在矩阵形式下推导过等而有所不同。如果这些记号看起来与您过去接触深度网络时看到的类似，并且在9月8日的讲座后更容易理解，那么就背景知识而言已经足够了（毕竟，深度学习系统的_核心_在一定程度上就在于我们无需为这些手动计算而烦恼）。但如果这些概念对您来说_完全_陌生，那么可能需要在选修本课程之前先修一门关于机器学习与神经网络的单独课程，或者至少要意识到本课程将需要大量的补课工作。

现在，利用这些梯度，在 `src/simple_ml.py` 文件中编写

In [11]:
!python3 -m pytest -k "nn_epoch"

============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0
rootdir: /home/x1879/dlsys/hw0
collected 6 items / 5 deselected / 1 selected                                  

tests/test_simple_ml.py .                                                [100%]

======================= 1 passed, 5 deselected in 3.77s ========================


And finally submit for autograding.

In [ ]:
!python3 -m mugrade submit YOUR_GRADER_KEY_HERE hw0 -k "nn_epoch"

### 训练一个完整的神经网络

和之前一样，尽管对通过自动评分器来说并非严格必要，但看看你能否很好地使用自己的神经网络函数来训练 MNIST 分类器其实挺有意思的。与 softmax 回归的情况类似，在 `simple_ml.py` 文件中有一个 `train_nn()` 函数，你可以用它通过多轮随机梯度下降来训练这个两层网络。例如，以下是训练一个具有 400 个隐藏单元的两层网络的代码。

In [12]:
import sys

# Reload the simple_ml module which has been cached from the earlier experiment
import importlib
import simple_ml
importlib.reload(simple_ml)

sys.path.append("src/")
from simple_ml import train_nn, parse_mnist

X_tr, y_tr = parse_mnist("data/train-images-idx3-ubyte.gz", 
                         "data/train-labels-idx1-ubyte.gz")
X_te, y_te = parse_mnist("data/t10k-images-idx3-ubyte.gz",
                         "data/t10k-labels-idx1-ubyte.gz")
train_nn(X_tr, y_tr, X_te, y_te, hidden_dim=400, epochs=20, lr=0.2)

| Epoch | Train Loss | Train Err | Test Loss | Test Err |
|     0 |    0.15711 |   0.04807 |   0.16755 |  0.05110 |
|     1 |    0.09868 |   0.02952 |   0.11645 |  0.03610 |
|     2 |    0.07483 |   0.02187 |   0.09892 |  0.03170 |
|     3 |    0.05934 |   0.01675 |   0.08823 |  0.02850 |
|     4 |    0.04949 |   0.01370 |   0.08317 |  0.02640 |
|     5 |    0.04079 |   0.01117 |   0.07812 |  0.02400 |
|     6 |    0.03464 |   0.00910 |   0.07509 |  0.02340 |
|     7 |    0.03002 |   0.00747 |   0.07327 |  0.02330 |
|     8 |    0.02625 |   0.00647 |   0.07179 |  0.02270 |
|     9 |    0.02347 |   0.00568 |   0.07105 |  0.02190 |
|    10 |    0.02084 |   0.00467 |   0.06988 |  0.02140 |
|    11 |    0.01888 |   0.00400 |   0.06949 |  0.02110 |
|    12 |    0.01719 |   0.00333 |   0.06922 |  0.02060 |
|    13 |    0.01562 |   0.00288 |   0.06853 |  0.02030 |
|    14 |    0.01414 |   0.00233 |   0.06808 |  0.02030 |
|    15 |    0.01291 |   0.00203 |   0.06759 |  0.02020 |
|    16 |    0

This takes about 30 seconds to run on Colab for our implementation, and as seen above, it achieve an error of 1.89\% on MNIST.  Not bad for less than 20 lines of code or so...

## 问题6：C++实现Softmax回归

本次作业的最后一问要求你实现与问题4中相同的函数——一个运行单轮线性softmax回归的函数。但在这里，你将使用C++而非Python来实现。（严格来说，这里的实际实现更像是原始C语言，但我们借助[pybind11](https://pybind11.readthedocs.io)库来构建与Python的接口，pybind11库使用了一些C++特性，后续作业中你也会用它来完成C++和Python之间的交互。虽然还有其他替代方案，但pybind11库作为接口相对友好，因为它是一个仅头文件的库，允许你在单个C++源文件中完成整个Python/C++接口的实现。）

你需要在`src/simple_ml_ext.cpp`文件中完成实现。先来看一下文件中相关的部分。你的代码将具体实现在以下函数中：

```cpp
void softmax_regression_epoch_cpp(const float *X, const unsigned char *y, 
                                  float *theta, size_t m, size_t n, size_t k, 
                                  float lr, size_t batch)
{
    /**
     * softmax回归轮次代码的C++版本。该函数应对由X和y（以及大小m,n,k）定义的数据运行单个轮次，
     * 并原地修改theta。你的函数可能需要分配（然后删除）一些辅助数组来存储logits和梯度。
     * 
     * 参数：
     *     X (const float *): 指向X数据的指针，大小为m*n，以行主序（C风格）存储
     *     y (const unsigned char *): 指向y数据的指针，大小为m
     *     theta (float *): 指向theta数据的指针，大小为n*k，以行主序（C风格）存储
     *     m (size_t): 样本数量
     *     n (size_t): 输入维度
     *     k (size_t): 类别数量
     *     lr (float): 学习率 / SGD步长
     *     batch (int): SGD小批量大小
     * 
     * 返回：
     *     (无)
     */

    /// 你的代码将写在这里
    
    /// 代码结束
}
```

我们来详细解读一下这个函数的参数。该函数本质上与Python实现相对应，但由于我们操作的是数组数据的原始指针，而非任何高层“矩阵”数据结构，因此需要传递一些额外的参数。具体来说，`X`、`y`和`theta`是指向之前部分相应NumPy数组原始数据的指针；对于二维数组，它们按C风格（行主序）存储，这意味着$X$的第一行在`X`中按顺序作为最前面的字节存储，然后是第二行，依此类推（这与_列主序_排列相对，后者按顺序存储矩阵的第一_列_，然后是第二列等）。我们还假设数据中无填充；即第二行紧接第一行之后，没有额外的字节，例如为了对齐内存到某个边界而添加的字节（所有这些话题将在课程的后续讨论中提及，但目前暂且忽略）。当然，由于传入函数的只是原始数据，为了知道底层矩阵的实际大小，我们还需要将这些大小显式地传递给函数，这就是`m`、`n`和`k`参数的作用。

为说明如何访问数据，注意因为`X`代表一个行主序的$m \times n$矩阵，若要访问$X$的$(i,j)$元素（第$i$行第$j$列的元素），我们将使用索引：
```cpp
X[i*n + j]
```
即，因为`X`有$n$列，并按列顺序存储，我们需要通过访问`X[i*n]`元素来访问第$i$行；若要访问该行中的第$j$个元素，则使用上述表达式。同样的逻辑也适用于`theta`矩阵，但重要的是，因为`theta`是一个$n \times k$矩阵，访问其$(i,j)$元素您将使用索引：
```cpp
theta[i*k + j]
```
与Python不同，在C++中像这样直接访问内存时需要格外小心，并且要非常习惯于这类表示法（或者构建额外的数据结构来帮助您以更直观的方式访问，但本次作业您应该只使用原始索引
```cpp
PYBIND11_MODULE(simple_ml_ext, m) {
    m.def("softmax_regression_epoch_cpp", 
    	[](py::array_t<float, py::array::c_style> X, 
           py::array_t<unsigned char, py::array::c_style> y, 
           py::array_t<float, py::array::c_style> theta,
           float lr,
           int batch) {
        softmax_regression_epoch_cpp(
        	static_cast<const float*>(X.request().ptr),
            static_cast<const unsigned char*>(y.request().ptr),
            static_cast<float*>(theta.request().ptr),
            X.request().shape[0],
            X.request().shape[1],
            theta.request().shape[1],
            lr,
            batch
           );
    },
    py::arg("X"), py::arg("y"), py::arg("theta"), 
    py::arg("lr"), py::arg("batch"));
}
```
这个代码已在文件中提供给你，你不应做任何修改。但对于好奇的人来说，这段代码本质上只是从提供的输入中提取原始指针（通过 pybinds 的 numpy 接口），然后调用相应的 `softmax_regression_epoch_cpp` 函数。

以上述内容为背景，请实现 `softmax_regression_epoch_cpp`，使其完成与你的 Python 实现相同的更新。注意，因为你只是直接访问原始数据，所以你需要手动执行所有的矩阵-向量乘积，而不能依赖 numpy 来替你完成所有矩阵运算（**注意：本次作业不要使用像 Eigen 这样的外部矩阵库，而是自己编写乘法代码……这是一个相对简单的操作**）。完成之后，你可以使用以下命令来测试该实现。

In [28]:
!make
!python3 -m pytest -k "softmax_regression_epoch_cpp"

c++ -O3 -Wall -shared -std=c++11 -fPIC -undefined dynamic_lookup $(python3 -m pybind11 --includes) src/simple_ml_ext.cpp -o src/simple_ml_ext.so
============================= test session starts ==============================
platform darwin -- Python 3.7.3, pytest-4.3.1, py-1.8.0, pluggy-0.9.0
rootdir: /Users/zkolter/Dropbox/class/10-714/homework/hw0, inifile:
plugins: remotedata-0.3.1, openfiles-0.3.2, doctestplus-0.3.0, arraydiff-0.3
collected 6 items / 5 deselected / 1 selected                                  

tests/test_simple_ml.py .                                                [100%]

==================== 1 passed, 5 deselected in 0.49 seconds ====================


Note that unlike our code before, we had to actually compile the C++ extension in order to run an test it.  This will be needed whenever there are C++ components of your code, but in all such cases we will include Makefiles that should compile all the relevant functions and include the necessary libraries/headers.  And finally, let's also submit the results to mugrade.

In [ ]:
!python3 -m mugrade submit YOUR_GRADER_KEY_HERE hw0 -k "softmax_regression_epoch_cpp"

### Training a full softmax regression classifier with the C++ version

Let's finally try training the whole softmax regression classifier using our "direct memory acesss" C++ version.  If the previous Python version took ~3 seconds, this should be blazing fast, right?

In [30]:
import sys
sys.path.append("src/")

# Reload the simple_ml module to include the newly-compiled C++ extension
import importlib
import simple_ml
importlib.reload(simple_ml)

from simple_ml import train_softmax, parse_mnist

X_tr, y_tr = parse_mnist("data/train-images-idx3-ubyte.gz", 
                         "data/train-labels-idx1-ubyte.gz")
X_te, y_te = parse_mnist("data/t10k-images-idx3-ubyte.gz",
                         "data/t10k-labels-idx1-ubyte.gz")

train_softmax(X_tr, y_tr, X_te, y_te, epochs=10, lr = 0.2, batch=100, cpp=True)

| Epoch | Train Loss | Train Err | Test Loss | Test Err |
|     0 |    0.35134 |   0.10182 |   0.33588 |  0.09400 |
|     1 |    0.32142 |   0.09268 |   0.31086 |  0.08730 |
|     2 |    0.30802 |   0.08795 |   0.30097 |  0.08550 |
|     3 |    0.29987 |   0.08532 |   0.29558 |  0.08370 |
|     4 |    0.29415 |   0.08323 |   0.29215 |  0.08230 |
|     5 |    0.28981 |   0.08182 |   0.28973 |  0.08090 |
|     6 |    0.28633 |   0.08085 |   0.28793 |  0.08080 |
|     7 |    0.28345 |   0.07997 |   0.28651 |  0.08040 |
|     8 |    0.28100 |   0.07923 |   0.28537 |  0.08010 |
|     9 |    0.27887 |   0.07847 |   0.28442 |  0.07970 |


As expected, the numbers match exactly our Python version, and the code is ... about 5 times slower?!  What is going on here?  Well, it turns out that the "manual" matrix multiplication code you probably wrote for the C++ version is extremely inefficient.  While Python itself is a slow, interpreted language, numpy itself is backed by matrix multiplications written in C (or, more likely, Fortran, believe it or not), that have been highly optmized to make use of vector operations, the cache hierarchy of different processors, and other features that are essential for efficient numerical operations.  We will cover these details much more in later lectures, and you'll even write a matrix library that can actually perform these operations relatively efficiently (at least for some special cases ... it's honestly not that easy to beat numpy in general).

But for now, assuming your code recreates the Python behavior, you're done with the assignment, and can get ready for our next dive into automatic differentiation.